In [1]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# Portable project-root detection
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent

elif (CURRENT_DIR / "notebooks").exists():
    PROJECT_DIR = CURRENT_DIR

else:
    PROJECT_DIR = None

    for parent in [CURRENT_DIR] + list(CURRENT_DIR.parents):
        if (parent / "notebooks").exists() and (parent / "requirements.txt").exists():
            PROJECT_DIR = parent
            break

    if PROJECT_DIR is None:
        raise FileNotFoundError(
            "Project root could not be detected. "
            "Please start JupyterLab from the CyberXAI-CSE-IDS2018 project folder."
        )


# ============================================================
# Project directories
# ============================================================

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
DOCUMENTATION_DIR = PROJECT_DIR / "documentation"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCUMENTATION_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Locate raw Parquet files
# ============================================================

raw_files = sorted(RAW_DIR.glob("*.parquet"))

if not raw_files:
    raise FileNotFoundError(
        f"No Parquet files found in:\n{RAW_DIR}\n\n"
        "Download the 10 dataset files from the GitHub dataset-v1 release "
        "and place them inside data/raw/."
    )

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Raw files found:", len(raw_files))


Project directory: D:\Sami Data Set\CyberXAI-CSE-IDS2018
Raw data directory: D:\Sami Data Set\CyberXAI-CSE-IDS2018\data\raw
Raw files found: 10


In [2]:
source_file_mapping = pd.DataFrame({
    "source_file_id": range(1, len(raw_files) + 1),
    "source_file_name": [file.name for file in raw_files]
})

display(source_file_mapping)

source_file_mapping.to_csv(
    DOCUMENTATION_DIR / "source_file_mapping.csv",
    index=False
)

,source_file_id,source_file_name
0,1,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...
1,2,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...
2,3,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...
3,4,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...
4,5,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...
5,6,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...
6,7,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...
7,8,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...
8,9,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...
9,10,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...


In [3]:
preprocessing_records = []

for source_file_id, file_path in enumerate(raw_files, start=1):
    print(
        f"[{source_file_id}/{len(raw_files)}] "
        f"Processing {file_path.name}",
        flush=True
    )

    try:
        df = pd.read_parquet(file_path)
        rows_before = len(df)
        columns_before = len(df.columns)

        # Standardise column names
        df.columns = df.columns.astype(str).str.strip()

        if "Label" not in df.columns:
            raise KeyError("Label column was not found.")

        # Preserve the original attack category
        df = df.rename(
            columns={"Label": "original_attack_label"}
        )

        df["original_attack_label"] = (
            df["original_attack_label"]
            .astype("string")
            .str.strip()
        )

        # Create the binary target:
        # 0 = benign, 1 = malicious
        df["binary_label"] = np.where(
            df["original_attack_label"]
            .str.casefold()
            .eq("benign"),
            0,
            1
        ).astype("int8")

        # Add traceability fields.
        # These must not be used as model predictors.
        df["source_file_id"] = np.int8(source_file_id)
        df["source_row_id"] = np.arange(
            len(df),
            dtype=np.int64
        )

        benign_count = int((df["binary_label"] == 0).sum())
        malicious_count = int((df["binary_label"] == 1).sum())

        output_path = (
            PROCESSED_DIR
            / f"{file_path.stem}_processed.parquet"
        )

        df.to_parquet(
            output_path,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        preprocessing_records.append({
            "source_file_id": source_file_id,
            "source_file_name": file_path.name,
            "processed_file_name": output_path.name,
            "rows_before": rows_before,
            "rows_after": len(df),
            "columns_before": columns_before,
            "columns_after": len(df.columns),
            "missing_values_removed": 0,
            "infinite_values_removed": 0,
            "duplicates_removed": 0,
            "benign_records": benign_count,
            "malicious_records": malicious_count,
            "status": "Completed",
            "error": ""
        })

        print(
            f"Saved | Rows: {len(df):,} | "
            f"Benign: {benign_count:,} | "
            f"Malicious: {malicious_count:,}",
            flush=True
        )

        del df
        gc.collect()

    except Exception as error:
        preprocessing_records.append({
            "source_file_id": source_file_id,
            "source_file_name": file_path.name,
            "processed_file_name": "",
            "rows_before": None,
            "rows_after": None,
            "columns_before": None,
            "columns_after": None,
            "missing_values_removed": 0,
            "infinite_values_removed": 0,
            "duplicates_removed": 0,
            "benign_records": None,
            "malicious_records": None,
            "status": "Error",
            "error": str(error)
        })

        print("Error:", error, flush=True)

preprocessing_log = pd.DataFrame(preprocessing_records)

display(preprocessing_log)

[1/10] Processing Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
Saved | Rows: 771,587 | Benign: 627,052 | Malicious: 144,535
[2/10] Processing Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet
Saved | Rows: 619,346 | Benign: 525,245 | Malicious: 94,101
[3/10] Processing DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet
Saved | Rows: 954,846 | Benign: 379,482 | Malicious: 575,364
[4/10] Processing DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet
Saved | Rows: 561,396 | Benign: 360,805 | Malicious: 200,591
[5/10] Processing DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet
Saved | Rows: 794,812 | Benign: 743,498 | Malicious: 51,314
[6/10] Processing DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet
Saved | Rows: 591,873 | Benign: 446,619 | Malicious: 145,254
[7/10] Processing Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet
Saved | Rows: 456,873 | Benign: 400,424 | Malicious: 56,449
[8/10] Processing Infil2-Th

,source_file_id,source_file_name,processed_file_name,rows_before,rows_after,columns_before,columns_after,missing_values_removed,infinite_values_removed,duplicates_removed,benign_records,malicious_records,status,error
0,1,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,771587,771587,78,81,0,0,0,627052,144535,Completed,
1,2,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,619346,619346,78,81,0,0,0,525245,94101,Completed,
2,3,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,954846,954846,78,81,0,0,0,379482,575364,Completed,
3,4,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,561396,561396,78,81,0,0,0,360805,200591,Completed,
4,5,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,794812,794812,78,81,0,0,0,743498,51314,Completed,
5,6,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,591873,591873,78,81,0,0,0,446619,145254,Completed,
6,7,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,456873,456873,78,81,0,0,0,400424,56449,Completed,
7,8,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,249170,249170,78,81,0,0,0,187136,62034,Completed,
8,9,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,830224,830224,78,81,0,0,0,829883,341,Completed,
9,10,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,829405,829405,78,81,0,0,0,828864,541,Completed,


In [4]:
preprocessing_log.to_csv(
    DOCUMENTATION_DIR / "preprocessing_log.csv",
    index=False
)

print("Preprocessing log saved successfully.")


Preprocessing log saved successfully.


In [5]:
processed_files = sorted(
    PROCESSED_DIR.glob("*_processed.parquet")
)

verification_records = []

for file_path in processed_files:
    parquet_file = pq.ParquetFile(file_path)

    verification_records.append({
        "file_name": file_path.name,
        "rows": parquet_file.metadata.num_rows,
        "columns": len(parquet_file.schema.names),
        "has_original_attack_label":
            "original_attack_label"
            in parquet_file.schema.names,
        "has_binary_label":
            "binary_label"
            in parquet_file.schema.names,
        "has_source_file_id":
            "source_file_id"
            in parquet_file.schema.names,
        "has_source_row_id":
            "source_row_id"
            in parquet_file.schema.names
    })

processed_verification = pd.DataFrame(
    verification_records
)

display(processed_verification)

print(
    "\nTotal processed rows:",
    f"{processed_verification['rows'].sum():,}"
)



,file_name,rows,columns,has_original_attack_label,has_binary_label,has_source_file_id,has_source_row_id
0,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,771587,81,True,True,True,True
1,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,619346,81,True,True,True,True
2,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,954846,81,True,True,True,True
3,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,561396,81,True,True,True,True
4,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,794812,81,True,True,True,True
5,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,591873,81,True,True,True,True
6,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,456873,81,True,True,True,True
7,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,249170,81,True,True,True,True
8,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,830224,81,True,True,True,True
9,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,829405,81,True,True,True,True



Total processed rows: 6,659,532


In [6]:
print("Total benign records:")
print(f"{preprocessing_log['benign_records'].sum():,}")

print("\nTotal malicious records:")
print(f"{preprocessing_log['malicious_records'].sum():,}")

print("\nTotal processed records:")
print(f"{preprocessing_log['rows_after'].sum():,}")

Total benign records:
5,329,008

Total malicious records:
1,330,524

Total processed records:
6,659,532


## Preprocessing Conclusion

The ten raw Parquet files were processed separately to limit memory
usage and preserve the original files. The data-quality audit found no
missing values, infinite values or within-file duplicate records;
therefore, no observations were imputed or removed.

The original `Label` field was retained as
`original_attack_label`, and a new binary target was created in which
0 represents benign traffic and 1 represents malicious traffic.
Source-file and source-row identifiers were added solely for
traceability and will be excluded from all model predictors.

The processed dataset contains 6,659,532 records, including 5,329,008
benign records and 1,330,524 malicious records.